# Visualize Correlations: mQTL Results
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.util import ceil_to_decimal, floor_to_decimal

## Visualize mQTL

In [ ]:
filetype = "pdf"
input_dirpath = PROCESSED_DATA_DIR / "REDS"
output_dirpath = REPORTS_DIR / "REDS" / "mQTL"
output_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
input_filename = make_filename("REDS", "Index", f"Genomics-{GENE_ID}", "mQTL")
df_mQTL_all = load_dataframe_from_file(input_dirpath / input_filename)

df_mQTL_all["RSID"] = df_mQTL_all["ID"].str.split(":", n=1, expand=True)[0]

df_mQTL_wide_pval = df_mQTL_all.pivot(columns="RSID", index="Metabolite", values="-log10(p-value)")
# Sort by rows, then by columns
df_mQTL_wide_pval = df_mQTL_wide_pval.sort_values(
    by=list(df_mQTL_wide_pval.index), axis=1, ascending=False
)
df_mQTL_wide_pval = df_mQTL_wide_pval.sort_values(
    by=list(df_mQTL_wide_pval.columns), axis=0, ascending=False
)
# Revert back to tall data to have NaN values and have all entries exist for all SNPs
df_mQTL_tall = df_mQTL_wide_pval.reset_index(drop=False).melt(
    id_vars="Metabolite", value_name="-log10(p-value)"
)
df_mQTL_tall = df_mQTL_tall.merge(
    df_mQTL_all[["RSID", "Metabolite", "Beta"]].drop_duplicates(),
    left_on=["RSID", "Metabolite"],
    right_on=["RSID", "Metabolite"],
    how="left",
)
all_counts = df_mQTL_wide_pval.count()
all_counts = all_counts.sort_values(ascending=False)
all_counts.head(10)

In [ ]:
num_metabolites_threshold = all_counts.iloc[1]
display_pvalues = False
display_beta = True

cmap_name = "viridis"
figsize_dim = round(all_counts.iloc[0] / 6)
zorder = 2
edgecolor = "xkcd:black"
linewidth = 1
width = 0.75
xlim_pads = (0, 0.01)
label_fontdict = {"size": "large"}
title_fontdict = {"size": "xx-large", "weight": "bold"}
tick_fontdict = {"size": "small"}
grid_kwargs = dict(
    linestyle="-",
    color="xkcd:light grey",
    which="both",
    zorder=1,
)
beta_kwargs = dict(
    xmajor_tick_multiple=0.2,
    xdecimal=2,
    xscalar=10,
    zorder=zorder,
    edgecolor=edgecolor,
    linewidth=linewidth,
    width=width,
    zero_line_kwargs=dict(
        linestyle="-",
        linewidth=linewidth,
        color=edgecolor,
        zorder=zorder,
    ),
)
pvalue_kwargs = dict(
    xmajor_tick_multiple=20,
    xdecimal=-1,
    xscalar=1,
    zorder=zorder,
    edgecolor=edgecolor,
    linewidth=linewidth,
    width=width,
)

beta_key, beta_label = ("Beta", "Effect Size")
mets_key, mets_label = ("Metabolite", None)
pvalue_key, pvalue_label = ("-log10(p-value)", "-log$_{10}$($p$-value)")


ordered_mets = df_mQTL_wide_pval.index.tolist()
counts = all_counts[all_counts >= num_metabolites_threshold].copy()
counts.head(10)

### Plot -log10(p-value) and/or effect size
#### Plot all mQTLs above metabolite threshold
##### All in one plot

In [ ]:
assert not (display_beta == display_pvalues == False), "At least one plot must be set to display."
vmin = 0
vmax = ceil_to_decimal(df_mQTL_tall[pvalue_key].max(), decimals=0)
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=cmap_name, norm=norm)
nrows = len(counts.index)
ncols = sum([display_pvalues, display_beta])
fig = plt.figure(figsize=(figsize_dim * ncols, figsize_dim * nrows))
subfigs = fig.subfigures(nrows, 1)

for RSID, subfig in zip(counts.index, subfigs):
    axes = subfig.subplots(1, ncols, sharex=False, sharey=True)
    axes = np.atleast_1d(axes)
    title = f"{RSID} mQTL"
    match len(axes):
        case 1 if display_pvalues:
            ax_pvalue, ax_beta = axes[0], None
            ax_pvalue.set_title(title, fontdict=title_fontdict)
        case 1 if display_beta:
            ax_pvalue, ax_beta = None, axes[0]
            ax_beta.set_title(title, fontdict=title_fontdict)
        case 2:
            ax_pvalue, ax_beta = axes
            subfig.suptitle(title, **title_fontdict)

    df_mQTL_snp = df_mQTL_tall[df_mQTL_tall["RSID"] == RSID].drop("RSID", axis=1)
    df_mQTL_snp = (
        df_mQTL_snp.set_index(mets_key).loc[ordered_mets].reset_index(drop=False).fillna(0)
    )
    colors = {i: sm.to_rgba(i) for z, i in df_mQTL_snp[pvalue_key].items()}

    if display_pvalues:
        ax_pvalue = sns.barplot(
            data=df_mQTL_snp,
            x=pvalue_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_pvalue,
            palette=colors,
            legend=False,
            zorder=pvalue_kwargs.get("zorder", zorder),
            edgecolor=pvalue_kwargs.get("edgecolor", edgecolor),
            linewidth=pvalue_kwargs.get("linewidth", linewidth),
            width=pvalue_kwargs.get("width", zorder),
        )
        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_tall[pvalue_key].min(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_tall[pvalue_key].max(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )
        ax_pvalue.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_pvalue.set_xlabel(pvalue_label, fontdict=label_fontdict)
        ax_pvalue.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_pvalue.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_pvalue.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_pvalue.xaxis.grid(**grid_kwargs)
        # Y-axis
        ax_pvalue.set_ylim((df_mQTL_snp.index.size, -1))
        ax_pvalue.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_pvalue.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_pvalue.yaxis.grid(**grid_kwargs)
        sns.despine(ax=ax_pvalue)

    if display_beta:
        ax_beta = sns.barplot(
            data=df_mQTL_snp,
            x=beta_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_beta,
            palette=colors,
            legend=False,
            zorder=beta_kwargs.get("zorder", zorder),
            edgecolor=beta_kwargs.get("edgecolor", edgecolor),
            linewidth=beta_kwargs.get("linewidth", linewidth),
            width=beta_kwargs.get("width", zorder),
        )
        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_tall[beta_key].min(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_tall[beta_key].max(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )
        ax_beta.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_beta.set_xlabel(beta_label, fontdict=label_fontdict)
        ax_beta.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_beta.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_beta.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_beta.xaxis.grid(**grid_kwargs)
        # Y-axis
        ax_beta.set_ylim((df_mQTL_snp.index.size, -1))
        ax_beta.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_beta.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_beta.yaxis.grid(**grid_kwargs)
        # Zero line
        if xmin < 0 and xmax > 0:
            ax_beta.vlines(
                0, ymin=-1, ymax=df_mQTL_snp.index.size, **beta_kwargs.get("zero_line_kwargs", {})
            )
        sns.despine(ax=ax_beta)


cax = fig.add_axes([1.05, 0.4, 0.025, 0.2])
fig.colorbar(sm, cax=cax, orientation="vertical")
cax.set_title(pvalue_label, fontdict=label_fontdict, ha="center")

cbar_tick_int = [x for x in range(1, int(vmax)) if vmax % x == 0][-1]
cax.set_yticks(np.arange(0, vmax + cbar_tick_int, cbar_tick_int))
cax.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=5)

for ticklabel in cax.get_yticklabels():
    ticklabel.set_horizontalalignment("center")

filename_args = [
    GENE_ID,
    f"geq{num_metabolites_threshold}",
]
if display_pvalues:
    filename_args += ["pvalue"]
if display_beta:
    filename_args += ["beta"]

output_filename = make_filename(*filename_args, filetype=filetype)
output_filepath = output_dirpath / output_filename
# fig.savefig(output_filepath, bbox_inches="tight")
logger.info(f"Saved file at: '{output_filepath.relative_to(PROJ_ROOT)}'.")

#### As individual plots
##### Constant across plots
Note: x-axis, y-axis, and colors are constant across plots

In [ ]:
assert not (display_beta == display_pvalues == False), "At least one plot must be set to display."
vmin = 0
vmax = ceil_to_decimal(df_mQTL_tall[pvalue_key].max(), decimals=0)
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=cmap_name, norm=norm)
nrows = 1
ncols = sum([display_pvalues, display_beta])


for RSID in counts.index:
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(figsize_dim * ncols, figsize_dim * nrows), sharey=True
    )
    axes = np.atleast_1d(axes)
    title = f"{RSID} mQTL"
    match len(axes):
        case 1 if display_pvalues:
            ax_pvalue, ax_beta = axes[0], None
            ax_pvalue.set_title(title, fontdict=title_fontdict)
        case 1 if display_beta:
            ax_pvalue, ax_beta = None, axes[0]
            ax_beta.set_title(title, fontdict=title_fontdict)
        case 2:
            ax_pvalue, ax_beta = axes
            fig.suptitle(title, **title_fontdict)

    df_mQTL_snp = df_mQTL_tall[df_mQTL_tall["RSID"] == RSID].drop("RSID", axis=1)
    df_mQTL_snp = (
        df_mQTL_snp.set_index(mets_key).loc[ordered_mets].reset_index(drop=False).fillna(0)
    )
    colors = {i: sm.to_rgba(i) for z, i in df_mQTL_snp[pvalue_key].items()}

    if display_pvalues:
        ax_pvalue = sns.barplot(
            data=df_mQTL_snp,
            x=pvalue_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_pvalue,
            palette=colors,
            legend=False,
            zorder=pvalue_kwargs.get("zorder", zorder),
            edgecolor=pvalue_kwargs.get("edgecolor", edgecolor),
            linewidth=pvalue_kwargs.get("linewidth", linewidth),
            width=pvalue_kwargs.get("width", zorder),
        )
        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_tall[pvalue_key].min(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_tall[pvalue_key].max(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )

        ax_pvalue.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_pvalue.set_xlabel(pvalue_label, fontdict=label_fontdict)
        ax_pvalue.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_pvalue.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_pvalue.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_pvalue.xaxis.grid(**grid_kwargs)

        # Y-axis
        ax_pvalue.set_ylim((df_mQTL_snp.index.size, -1))
        ax_pvalue.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_pvalue.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_pvalue.yaxis.grid(**grid_kwargs)
        sns.despine(ax=ax_pvalue)

    if display_beta:
        ax_beta = sns.barplot(
            data=df_mQTL_snp,
            x=beta_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_beta,
            palette=colors,
            legend=False,
            zorder=beta_kwargs.get("zorder", zorder),
            edgecolor=beta_kwargs.get("edgecolor", edgecolor),
            linewidth=beta_kwargs.get("linewidth", linewidth),
            width=beta_kwargs.get("width", zorder),
        )

        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_tall[beta_key].min(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_tall[beta_key].max(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )

        ax_beta.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_beta.set_xlabel(beta_label, fontdict=label_fontdict)
        ax_beta.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_beta.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_beta.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_beta.xaxis.grid(**grid_kwargs)

        # Y-axis
        ax_beta.set_ylim((df_mQTL_snp.index.size, -1))
        ax_beta.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_beta.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_beta.yaxis.grid(**grid_kwargs)
        # Zero line
        if xmin < 0 and xmax > 0:
            ax_beta.vlines(
                0, ymin=-1, ymax=df_mQTL_snp.index.size, **beta_kwargs.get("zero_line_kwargs", {})
            )
        sns.despine(ax=ax_beta)

    filename_args = [
        GENE_ID,
        f"geq{num_metabolites_threshold}",
        RSID,
    ]
    if display_pvalues:
        filename_args += ["pvalue"]
    if display_beta:
        filename_args += ["beta"]

    output_filename = make_filename(*filename_args, filetype=filetype)
    output_filepath = output_dirpath / output_filename
    fig.savefig(output_filepath, bbox_inches="tight")
    logger.info(f"Saved file at: '{output_filepath.relative_to(PROJ_ROOT)}'.")

##### Not constant across plots
Note: x-axis, y-axis, and colors are not constant across plots

In [ ]:
assert not (display_beta == display_pvalues == False), "At least one plot must be set to display."
output_dirpath_all = output_dirpath / "individual-rsids"
output_dirpath_all.mkdir(exist_ok=True, parents=True)

nrows = 1
ncols = sum([display_pvalues, display_beta])


for RSID in counts.index:
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(figsize_dim * ncols, figsize_dim * nrows), sharey=True
    )
    axes = np.atleast_1d(axes)
    title = f"{RSID} mQTL"
    match len(axes):
        case 1 if display_pvalues:
            ax_pvalue, ax_beta = axes[0], None
            ax_pvalue.set_title(title, fontdict=title_fontdict)
        case 1 if display_beta:
            ax_pvalue, ax_beta = None, axes[0]
            ax_beta.set_title(title, fontdict=title_fontdict)
        case 2:
            ax_pvalue, ax_beta = axes
            fig.suptitle(title, **title_fontdict)

    df_mQTL_snp = df_mQTL_tall[df_mQTL_tall["RSID"] == RSID].drop("RSID", axis=1)
    df_mQTL_snp = (
        df_mQTL_snp.set_index(mets_key).dropna().sort_values(by=pvalue_key, ascending=False)
    )

    if display_pvalues:
        ax_pvalue = sns.barplot(
            data=df_mQTL_snp,
            x=pvalue_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_pvalue,
            palette=cmap_name,
            legend=False,
            zorder=pvalue_kwargs.get("zorder", zorder),
            edgecolor=pvalue_kwargs.get("edgecolor", edgecolor),
            linewidth=pvalue_kwargs.get("linewidth", linewidth),
            width=pvalue_kwargs.get("width", zorder),
        )
        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_snp[pvalue_key].min(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_snp[pvalue_key].max(),
            decimals=pvalue_kwargs.get("xdecimal", 0),
            scalar=pvalue_kwargs.get("xscalar", 1),
        )

        ax_pvalue.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_pvalue.set_xlabel(pvalue_label, fontdict=label_fontdict)
        ax_pvalue.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_pvalue.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(pvalue_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_pvalue.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_pvalue.xaxis.grid(**grid_kwargs)
        # Y-axis
        ax_pvalue.set_ylim((df_mQTL_snp.index.size, -1))
        ax_pvalue.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_pvalue.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_pvalue.yaxis.grid(**grid_kwargs)
        sns.despine(ax=ax_pvalue)

    if display_beta:
        ax_beta = sns.barplot(
            data=df_mQTL_snp,
            x=beta_key,
            y=mets_key,
            hue=pvalue_key,
            ax=ax_beta,
            palette=cmap_name,
            legend=False,
            zorder=beta_kwargs.get("zorder", zorder),
            edgecolor=beta_kwargs.get("edgecolor", edgecolor),
            linewidth=beta_kwargs.get("linewidth", linewidth),
            width=beta_kwargs.get("width", zorder),
        )

        # X-axis
        xmin = floor_to_decimal(
            df_mQTL_snp[beta_key].min(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )
        xmax = ceil_to_decimal(
            df_mQTL_snp[beta_key].max(),
            decimals=beta_kwargs.get("xdecimal", 0),
            scalar=beta_kwargs.get("xscalar", 1),
        )
        ax_beta.set_xlim((xmin * (1 + xlim_pads[0]), xmax * (1 + xlim_pads[-1])))
        ax_beta.set_xlabel(beta_label, fontdict=label_fontdict)
        ax_beta.xaxis.set_major_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10))
        )
        ax_beta.xaxis.set_minor_locator(
            mpl.ticker.MultipleLocator(beta_kwargs.get("xmajor_tick_multiple", 10) / 2)
        )
        ax_beta.xaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"))
        ax_beta.xaxis.grid(**grid_kwargs)
        # Y-axis
        ax_beta.set_ylim((df_mQTL_snp.index.size, -1))
        ax_beta.set_ylabel(mets_label, fontdict=label_fontdict)
        ax_beta.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=1)
        ax_beta.yaxis.grid(**grid_kwargs)
        # Zero line
        if xmin < 0 and xmax > 0:
            ax_beta.vlines(
                0, ymin=-1, ymax=len(ordered_mets), **beta_kwargs.get("zero_line_kwargs", {})
            )
        sns.despine(ax=ax_beta)
    cax = fig.add_axes([1.0, 0.4, 0.025, 0.2])
    vmax = ceil_to_decimal(df_mQTL_snp[pvalue_key].max(), decimals=0, scalar=1)
    fig.colorbar(
        plt.cm.ScalarMappable(cmap=cmap_name, norm=mpl.colors.Normalize(vmin=0, vmax=vmax)),
        cax=cax,
        orientation="vertical",
    )
    cax.set_title(pvalue_label, fontdict=label_fontdict, ha="center")

    cbar_tick_int = [x for x in range(1, int(vmax)) if vmax % x == 0]
    if len(cbar_tick_int) > 1:
        cbar_tick_int = cbar_tick_int[-1]
        cax.set_yticks(np.arange(0, vmax + cbar_tick_int, cbar_tick_int))
    else:
        cax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
    cax.yaxis.set_tick_params(labelsize=tick_fontdict.get("size", "small"), pad=5)

    for ticklabel in cax.get_yticklabels():
        ticklabel.set_horizontalalignment("center")

    filename_args = [GENE_ID, RSID]
    if display_pvalues:
        filename_args += ["pvalue"]
    if display_beta:
        filename_args += ["beta"]

    output_filename = make_filename(*filename_args, filetype=filetype)
    output_filepath = output_dirpath_all / output_filename
    fig.savefig(output_filepath, bbox_inches="tight")
    plt.show()
    plt.close()
    logger.info(f"Saved file at: '{output_filepath.relative_to(PROJ_ROOT)}'.")